In [ ]:
import os
import copernicusmarine

# Locally: interactive. On GCP: set these env vars from Secret Manager.
# COPERNICUSMARINE_SERVICE_USERNAME / COPERNICUSMARINE_SERVICE_PASSWORD
copernicusmarine.login()  # prompts once, stores creds locally

/Users/claytonpreston/Documents/NextWave/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO - 2026-11-17T14:27:24Z - Using existing credentials from /Users/claytonpreston/.copernicusmarine/.copernicusmarine-credentials. Use --force-overwrite combined with credentials (specified by arguments, netrc file or environment variables) to always overwrite.


True

In [ ]:
# sources.py
import json, numpy as np, pandas as pd, requests, xarray as xr, copernicusmarine

WAVE_VARS = ["VHM0", "VTPK", "VTM10", "VMDR",
             "VHM0_WW", "VTM01_WW", "VMDR_WW",
             "VHM0_SW1", "VTM01_SW1", "VMDR_SW1",
             "VHM0_SW2", "VTM01_SW2", "VMDR_SW2"]       # check which exist: ds.data_vars

def download_waves(dataset_id, bbox, start, end, path="data/waves.nc"):
    copernicusmarine.subset(dataset_id=dataset_id, variables=WAVE_VARS,
        start_datetime=start, end_datetime=end, output_filename="waves.nc",
        output_directory="data", overwrite=True, **bbox)
    return xr.open_dataset(path).load()

def records(df):
    """UTC ISO times, NaN -> null. The one place formatting happens."""
    df = df.copy()
    df.index = pd.to_datetime(df.index, utc=True)
    df = df.reset_index(names="time")
    df["time"] = df["time"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")
    return json.loads(df.to_json(orient="records"))

def sea_cell(da2d, lat, lon):
    lats, lons = np.meshgrid(da2d.latitude.values, da2d.longitude.values, indexing="ij")
    d = np.hypot(lats - lat, (lons - lon) * np.cos(np.radians(lat)))
    d = np.where(np.isnan(da2d.values), np.inf, d)
    return np.unravel_index(d.argmin(), d.shape)

def wave_series(ds, lat, lon):
    vars_ = [v for v in WAVE_VARS if v in ds.data_vars]
    first = ds["VHM0"].isel(time=0)
    i, j = sea_cell(first, lat, lon)
    clat, clon = float(first.latitude[i]), float(first.longitude[j])
    dist_km = 111.2 * float(np.hypot(clat - lat, (clon - lon) * np.cos(np.radians(lat))))
    df = (ds[vars_].isel(latitude=i, longitude=j).to_dataframe()
            .drop(columns=["latitude", "longitude"], errors="ignore"))
    return dict(cell=dict(lat=clat, lon=clon, dist_km=dist_km), hourly=records(df))

def _om(url, lat, lon, hourly, **extra):
    r = requests.get(url, params=dict(latitude=lat, longitude=lon, hourly=hourly,
                     timezone="UTC", forecast_days=8, **extra), timeout=30)
    r.raise_for_status()
    return pd.DataFrame(r.json()["hourly"]).set_index("time")

WIND_MODELS = ["ecmwf_ifs025", "icon_eu", "gfs_seamless"]   # verify ids in Open-Meteo docs

def wind_by_model(lat, lon):
    cols = "wind_speed_10m,wind_direction_10m,wind_gusts_10m"
    out = {}
    for m in WIND_MODELS:
        df = _om("https://api.open-meteo.com/v1/forecast", lat, lon, cols,
                 wind_speed_unit="kn", models=m)
        out[m] = records(df.rename(columns={"wind_speed_10m": "wind_kn",
                 "wind_direction_10m": "wind_dir", "wind_gusts_10m": "gust_kn"}))
    return out

def atmos(lat, lon):
    return records(_om("https://api.open-meteo.com/v1/forecast", lat, lon,
        "temperature_2m,precipitation,cloud_cover,pressure_msl,weather_code"))

def marine_alt(lat, lon):   # second wave model + sea temperature (verify variable names)
    return records(_om("https://marine-api.open-meteo.com/v1/marine", lat, lon,
        "wave_height,wave_direction,wave_period,swell_wave_height,swell_wave_period,"
        "wind_wave_height,sea_surface_temperature"))

In [ ]:
# build_payload.py
def build_payload(SPOTS, ds, run_id, now_utc):
    spots = {}
    for sid, cfg in SPOTS.items():
        lat, lon = cfg["lat"], cfg["lon"]
        if not (ds.latitude.min() <= lat <= ds.latitude.max() and
                ds.longitude.min() <= lon <= ds.longitude.max()):
            print(f"skip {sid}: outside box"); continue
        spots[sid] = dict(
            config=cfg,
            waves=dict(copernicus=wave_series(ds, lat, lon), om_marine=marine_alt(lat, lon)),
            wind=wind_by_model(lat, lon),
            atmos=atmos(lat, lon),
        )
    return dict(schema=1, run=dict(id=run_id, generated_at=now_utc.isoformat() + "Z"), spots=spots)